In [1]:
"""
Credit Card Fraud Detection
----------------------------
Benchmarks Decision Tree, KNN, Logistic Regression, and SVM classifiers
on the highly imbalanced Kaggle Credit Card Fraud dataset.

Dataset: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
Place 'creditcard.csv' in the same folder as this script before running.

Techniques used:
- StandardScaler for feature scaling
- SMOTE for handling extreme class imbalance
- Evaluation via Accuracy, Precision, Recall, F1-score, and AUC
  (accuracy alone is misleading on imbalanced data)
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,f1_score, roc_auc_score, confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE

# Load dataset

In [2]:
df = pd.read_csv('creditcard.csv')
def load_data(path="creditcard.csv"):
    df = pd.read_csv(path)
    print(f"Dataset shape: {df.shape}")
    print(f"Fraud cases: {df['Class'].sum()} / {len(df)} "
          f"({100 * df['Class'].mean():.3f}%)")
    return df

In [3]:
df.columns

Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class'],
      dtype='object')

In [4]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0.0
1,0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0.0
2,1,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0.0
3,1,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0.0
4,2,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0.0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47628 entries, 0 to 47627
Data columns (total 31 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Time    47628 non-null  int64  
 1   V1      47628 non-null  float64
 2   V2      47628 non-null  float64
 3   V3      47628 non-null  float64
 4   V4      47628 non-null  float64
 5   V5      47628 non-null  float64
 6   V6      47628 non-null  float64
 7   V7      47628 non-null  float64
 8   V8      47628 non-null  float64
 9   V9      47628 non-null  float64
 10  V10     47628 non-null  float64
 11  V11     47628 non-null  float64
 12  V12     47628 non-null  float64
 13  V13     47628 non-null  float64
 14  V14     47628 non-null  float64
 15  V15     47628 non-null  float64
 16  V16     47628 non-null  float64
 17  V17     47627 non-null  float64
 18  V18     47627 non-null  float64
 19  V19     47627 non-null  float64
 20  V20     47627 non-null  float64
 21  V21     47627 non-null  float64
 22

In [6]:
df.describe()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,47628.000000,47628.000000,47628.000000,47628.000000,47628.000000,47628.000000,47628.000000,47628.000000,47628.000000,47628.000000,...,47627.000000,47627.000000,47627.000000,47627.000000,47627.000000,47627.000000,47627.000000,47627.000000,47627.000000,47627.000000
mean,28183.123562,-0.240703,0.021469,0.694689,0.191074,-0.249837,0.102310,-0.119867,0.053450,0.141204,...,-0.027059,-0.107091,-0.039399,0.007890,0.136314,0.022561,0.004907,0.004021,91.777163,0.003065
std,13001.722320,1.884794,1.626518,1.519894,1.402196,1.412854,1.309931,1.282101,1.216930,1.214104,...,0.737251,0.637200,0.579666,0.594175,0.438367,0.502411,0.388191,0.335275,249.818013,0.055283
min,0.000000,-56.407510,-72.715728,-32.965346,-5.172595,-42.147898,-26.160506,-26.548144,-41.484823,-9.283925,...,-20.262054,-8.593642,-26.751119,-2.836627,-7.495741,-1.577118,-8.567638,-9.617915,0.000000,0.000000
25%,20517.000000,-0.990001,-0.548684,0.220594,-0.714455,-0.857297,-0.636473,-0.602224,-0.146950,-0.597975,...,-0.231991,-0.528167,-0.179247,-0.322459,-0.128057,-0.329467,-0.063838,-0.006862,7.590000,0.000000
50%,32930.500000,-0.248789,0.085468,0.800243,0.195142,-0.282244,-0.152017,-0.074952,0.057092,0.026153,...,-0.069134,-0.081945,-0.051439,0.061661,0.175871,-0.068195,0.008665,0.021873,24.990000,0.000000
75%,38236.000000,1.156434,0.736933,1.432158,1.070576,0.286638,0.492728,0.425886,0.329612,0.837269,...,0.107256,0.306116,0.078475,0.401231,0.421741,0.302743,0.083933,0.076168,83.110000,0.000000
max,43282.000000,1.960497,18.183626,4.101716,16.491217,34.801666,22.529298,36.677268,20.007208,10.392889,...,22.614889,5.805795,17.297845,4.014444,5.525093,3.517346,11.135740,33.847808,12910.930000,1.000000


In [7]:
df.isnull().sum()

,0
Time,0
V1,0
V2,0
V3,0
V4,0
V5,0
V6,0
V7,0
V8,0
V9,0


# Preprocess: scale + train/test split + SMOTE

In [8]:
def preprocess(df, test_size=0.2, random_state=42):
    X = df.drop(columns=["Class"])
    y = df["Class"]

    # Scale 'Amount' and 'Time' (V1-V28 are already PCA-scaled)
    scaler = StandardScaler()
    X[["Amount", "Time"]] = scaler.fit_transform(X[["Amount", "Time"]])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, stratify=y, random_state=random_state)

    print(f"\nBefore SMOTE - Train fraud cases: {y_train.sum()} / {len(y_train)}")

    # Apply SMOTE only to the training set (never touch test data)
    smote = SMOTE(random_state=random_state)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    print(f"After SMOTE  - Train fraud cases: {y_train_res.sum()} / {len(y_train_res)}")

    return X_train_res, X_test, y_train_res, y_test

# Define models

In [9]:
def get_models():
    return {
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "SVM": SVC(kernel="rbf", probability=True, random_state=42),
    }

# Train, evaluate, and compare all models

In [10]:
def evaluate_models(models, X_train, X_test, y_train, y_test):
    results = []

    for name, model in models.items():
        print(f"\nTraining {name}...")
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") \
            else model.decision_function(X_test)

        metrics = {
            "Model": name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred),
            "F1-Score": f1_score(y_test, y_pred),
            "AUC": roc_auc_score(y_test, y_proba),
        }
        results.append(metrics)

        print(classification_report(y_test, y_pred, target_names=["Legit", "Fraud"]))
        print(f"AUC: {metrics['AUC']:.4f}")

        # Confusion matrix plot
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(4, 3))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"])
        plt.title(f"Confusion Matrix - {name}")
        plt.ylabel("Actual")
        plt.xlabel("Predicted")
        plt.tight_layout()
        plt.savefig(f"confusion_matrix_{name.replace(' ', '_')}.png")
        plt.close()

    return pd.DataFrame(results).sort_values(by="AUC", ascending=False)

# Main pipeline

In [11]:
if __name__ == "__main__":
    df = load_data("creditcard.csv")
    # Drop rows where 'Class' column has NaN values
    df.dropna(subset=['Class'], inplace=True)
    X_train, X_test, y_train, y_test = preprocess(df)

    models = get_models()
    results_df = evaluate_models(models, X_train, X_test, y_train, y_test)

    print("\n" + "=" * 60)
    print("FINAL MODEL COMPARISON")
    print("=" * 60)
    print(results_df.to_string(index=False))

    results_df.to_csv("model_comparison_results.csv", index=False)
    print("\nResults saved to model_comparison_results.csv")
    print("Confusion matrix plots saved as PNG files.")

    best_model = results_df.iloc[0]
    print(f"\nBest model: {best_model['Model']} "
          f"(AUC: {best_model['AUC']:.4f}, Precision: {best_model['Precision']:.4f})")

Dataset shape: (47628, 31)
Fraud cases: 146.0 / 47628 (0.307%)

Before SMOTE - Train fraud cases: 117.0 / 38101
After SMOTE  - Train fraud cases: 37984.0 / 75968

Training Decision Tree...
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00      9497
       Fraud       0.74      0.79      0.77        29

    accuracy                           1.00      9526
   macro avg       0.87      0.90      0.88      9526
weighted avg       1.00      1.00      1.00      9526

AUC: 0.8961

Training KNN...
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00      9497
       Fraud       0.61      0.97      0.75        29

    accuracy                           1.00      9526
   macro avg       0.80      0.98      0.87      9526
weighted avg       1.00      1.00      1.00      9526

AUC: 0.9825

Training Logistic Regression...
              precision    recall  f1-score   support

       Legit       1.00     